Nous allons construire quelques variables pertinentes, afin de tester une autre approche de celle faite lors du preprocessing.
Ainsi, nous verrons si ces nouvelles variables apportent réellement de l'information.

Pour cela nous utiliserons le dataset_clean qui provient du nettoyage de notre projet 2.

In [1]:
# les bibliothèques
import pandas as pd
import numpy as np


In [4]:
# Importons notre dataset_clean et faisons une copy

df_clean = pd.read_csv("../data/processed/healthcare_stroke_dataset_clean.csv")

df_features = df_clean.copy()

display(df_features.head())

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.1,never smoked,1
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


Nous créerons des variables à utilité médiacale puisque nous sommes dans des facteurs de santé médicale.

Nous avons vu lors de l'analyse explorative, trois facteurs ressortaient clairement :

l'âge, la glycémie, les antécédents cardiovasculaires et hypertension (age, avg_glucose_level, heart_disease, hypertension).

Nous allons donc commencer par eux.

#### Variable 1: senior 
Nous avons vu que le risque d'AVC augmente fortement avec l'âge.

Dans de nombreuses études épidémiologiques, 65 ans est souvent utilisé comme seuil pour définir une personne âgée.

In [25]:
# créeons la variable "senior"

df_features["senior"] = (df_features["age"] >= 65).astype(int)

df_features["senior"].value_counts()

senior
0    4082
1    1027
Name: count, dtype: int64

In [26]:
# Observons si les seniors présentent effectivement davantage d'AVC
pd.crosstab(
    df_features["senior"],
    df_features["stroke"],
    normalize = "index"
).round(3) *100

stroke,0,1
senior,,
0,97.8,2.2
1,84.5,15.5


##### Interprétation

On voit qu'un senior a un risque d'environ 7 fois plus élevé de faire un AVC : 15,5÷2,2 donne environ 7

ce qui fait une grande différence.

Cette variable est très pertinente pour certains modèles.

#### Variable 2 : high_glucose

Nous avons observé que les patients ayant eu un AVC avaient une glycémie moyenne plus élevée.

En pratique clinique, une glycémie à jeun ≥ 126 mg/dL est un seuil diagnostique du diabète. Notre variable n'est pas forcément une glycémie à jeun, mais ce seuil reste une référence intéressante.

In [27]:
# Créons la variable "high_glucose"

df_features["high_glucose"] = (df_features["avg_glucose_level"]>=126).astype(int)

df_features["high_glucose"].value_counts()


high_glucose
0    4129
1     980
Name: count, dtype: int64

In [29]:
# Observons si une glycémie élevée présentent effectivement davantage d'AVC
pd.crosstab(
            df_features["high_glucose"],
            df_features["stroke"],
            normalize = "index"
).round(3)*100

stroke,0,1
high_glucose,,
0,96.4,3.6
1,89.8,10.2


##### Interprétation

Nous remarquons que le risque est presque 3 fois plus élevé pour un patient qui a une glycémie élevée d'avoir un AVC.

Ce résultat est cohérent avec la littérature médicale : une glycémie élevée est un facteur de risque cardiovasculaire.


#### Variable 3 : cardio_risk

Une personne présentant : une hypertension, ou une maladie cardiaque,

présente déjà un risque cardiovasculaire accru.

In [37]:
# Créons la variable "cardio_risk"

df_features["cardio_risk"] = ( 
    (df_features["heart_disease"] == 1) | 
    (df_features["hypertension"] == 1) 
    ).astype(int)


In [39]:
df_features["cardio_risk"].value_counts()

cardio_risk
0    4399
1     710
Name: count, dtype: int64

In [38]:
# Observons si un risque cardiaque présentent effectivement davantage d'AVC
pd.crosstab(
            df_features["cardio_risk"],
            df_features["stroke"],
            normalize = "index"
).round(3)*100

stroke,0,1
cardio_risk,,
0,96.6,3.4
1,85.9,14.1


##### Interprétation

Le risque pour un patient ayant une maladie cardiovasculaire est environ 4 fois plus élevé.

C'est logique car : l'hypertension, la maladie cardiaque sont déjà des facteurs connus.

La variable cardio_risk résume ces deux informations en une seule.

Bien que cardio_risk donne une information redondante car elle est construite à partir de hypertension et heart_disease.
Les modèles modernes comme Random Forest, XGBoost... savent gérer ce type de redondance.